# Quadcopter Cascade PID — Figure-8 Trajectory Tracking

This notebook implements a **Cascade PID Controller** for quadcopter trajectory tracking using [c4dynamics](https://c4dynamics.github.io/c4dynamics/).

A nonlinear 6-DOF rigid body model tracks a figure-8 reference trajectory using three nested PID control loops running at different rates.

---

## What you can modify

| Section | What to change |
|---------|----------------|
| **Vehicle** | Mass, inertia, arm length |
| **Trajectory** | Figure-8 size, speed, altitude, duration |
| **Controller** | PID gains for each loop |

After modifying parameters, run all cells top to bottom.

---

## How to use this notebook

1. Edit parameters in **Section 1 — User Inputs**
2. Run **Section 2 — Run Simulation** (one cell)
3. View results in **Section 3 — Results**

All implementation details (dynamics, PID class, simulation loop) are in `quad_pid_utils.py`.

In [ ]:
# Install c4dynamics if running on Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install c4dynamics -q

from quad_pid_utils import run_fig8_pid, plot_results, compute_metrics

---
## Section 1 — User Inputs

Modify the three dictionaries below to configure your simulation.

In [ ]:
# ── Vehicle Parameters ──
# Physical properties of the quadcopter.
# Change these to match your specific vehicle.

vehicle = {
    'm'  : 1.0,     # mass [kg]
    'g'  : 9.81,    # gravity [m/s²]
    'Ixx': 0.0196,  # roll  moment of inertia [kg.m²]
    'Iyy': 0.0196,  # pitch moment of inertia [kg.m²]
    'Izz': 0.0264,  # yaw   moment of inertia [kg.m²]
    'l'  : 0.225,   # arm length — center to motor [m]
    'kT' : 1.0,     # thrust coefficient (normalized)
    'kQ' : 0.01,    # torque coefficient (normalized)
}

In [ ]:
# ── Trajectory Parameters ───
# Define the figure-8 shape, speed, and flight conditions.
#
# The figure-8 is defined by:
#   x(t) = A * sin(omega * t)
#   y(t) = B * sin(2 * omega * t)
#
# A cosine ramp is applied for the first 12 seconds of flight
# to avoid a velocity discontinuity at trajectory start.

trajectory = {
    'A'    : 1.5,   # figure-8 X amplitude [m]
    'B'    : 1.0,   # figure-8 Y amplitude [m]
    'omega': 0.3,   # angular frequency [rad/s]
                    # period = 2*pi/omega ≈ 20.9 s per cycle
    'z_ref': 1.5,   # constant hover altitude [m]
    't_end': 40.0,  # total simulation duration [s]
}

In [ ]:
# ── Controller Parameters (Advanced) ───
# Default gains are tuned for the default vehicle above.
# If you change the vehicle, you may need to retune these.
#
# Cascade loop structure:
#   Inner  loop (200 Hz) — angular rate  → torques
#   Middle loop (100 Hz) — attitude      → desired rates
#   Outer  loop ( 50 Hz) — position      → desired angles + thrust
#
# Tuning order: inner first, then middle, then outer.

controller = {

    # ── Inner loop — angular rate (200 Hz) ──
    # Controls P, Q, R body rates → outputs torques
    'Kp_p': 0.80,  'Ki_p': 0.0001,  'Kd_p': 0.010,   # roll  rate
    'Kp_q': 0.80,  'Ki_q': 0.0001,  'Kd_q': 0.010,   # pitch rate
    'Kp_r': 0.60,  'Ki_r': 0.0001,  'Kd_r': 0.008,   # yaw   rate

    # ── Middle loop — attitude (100 Hz) ──
    # Controls Phi, Theta, Psi Euler angles → outputs desired rates
    'Kp_phi'  : 7.0,  'Ki_phi'  : 0.0001,  'Kd_phi'  : 0.90,  # roll
    'Kp_theta': 7.0,  'Ki_theta': 0.0001,  'Kd_theta': 0.90,  # pitch
    'Kp_psi'  : 4.0,  'Ki_psi'  : 0.5,     'Kd_psi'  : 0.40,  # yaw

    # ── Outer loop — position (50 Hz) ──
    # Controls X, Y, Z position → outputs desired angles + thrust
    'Kp_x': 1.00,  'Ki_x': 0.01,  'Kd_x': 0.90,  # X position
    'Kp_y': 1.10,  'Ki_y': 0.01,  'Kd_y': 0.80,  # Y position
    'Kp_z': 10.0,  'Ki_z': 0.50,  'Kd_z': 1.50,  # altitude

    # ── Velocity feedforward gains ──
    # Added to position PID output to reduce phase lag.
    # Increase if actual path lags behind reference.
    # Decrease if actual path overshoots reference.
    'Kff_x': 0.2479,
    'Kff_y': 0.35,
}

# ── Simulation Settings ───
sim = {
    'dt'   : 0.005,              # master timestep [s] = inner loop rate
    't_end': trajectory['t_end'] # end time [s]
}

---
## Section 2 — Model Summary

### System Model

The quadcopter is modeled as a 12-state nonlinear rigid body:

$$\mathbf{X} = [x,\ y,\ z,\ \dot{x},\ \dot{y},\ \dot{z},\ \phi,\ \theta,\ \psi,\ p,\ q,\ r]^T$$

Four control inputs:

$$\mathbf{U} = [F,\ \tau_\phi,\ \tau_\theta,\ \tau_\psi]^T$$

The full nonlinear Newton-Euler equations are integrated numerically using RK45 at each timestep — no linearization or small-angle approximation is applied.

### Cascade PID Architecture

```
Position ref ──► Outer PID (50 Hz) ──► Attitude ref
                                             │
                               Middle PID (100 Hz) ──► Rate ref
                                                            │
                                           Inner PID (200 Hz) ──► Torques ──► Plant
```

Each controller implements:

$$u = K_p e + K_i \int e\, dt - K_d \dot{m}$$

where $\dot{m}$ is the **derivative of the measurement** (not the error), preventing derivative kick when the reference changes suddenly.

**Key features:**
- Derivative on measurement — no derivative kick at setpoint changes
- First-order derivative filter — reduces high-frequency noise amplification  
- Integrator anti-windup — prevents integrator accumulation during saturation
- Velocity feedforward — reduces phase lag at trajectory turns

---
## Section 3 — Run Simulation

In [ ]:
results = run_fig8_pid(vehicle, trajectory, controller, sim)
print('Simulation complete.')

---
## Section 4 — Results

In [ ]:
# Time history plots — position, Euler angles, control inputs
# 3D trajectory plot (steady state)
plot_results(results)

In [ ]:
# RMSE metrics — absolute and normalized
metrics = compute_metrics(results)